In [ ]:
#My Project

In [ ]:
from datasets import load_dataset

# Load the whole dataset
dataset = load_dataset('PKU-Alignment/BeaverTails')

# Load only the round 0 dataset
round0_dataset = load_dataset('PKU-Alignment/BeaverTails', data_dir='round0')

# Load the training dataset
train_dataset = load_dataset('PKU-Alignment/BeaverTails', split='330k_train')
test_dataset = load_dataset('PKU-Alignment/BeaverTails', split='330k_test')

In [ ]:
print(f"Shape of train_dataset: ({len(train_dataset)}, {len(train_dataset.column_names)})\n")
print(f"Column names of train_dataset: {train_dataset.column_names}\n")
print(f"Data types of train_dataset columns:\n{train_dataset.features}")
dataset_pd = train_dataset.to_pandas()
dataset_pd.head()

### Checking for Missing Values



In [ ]:
missing_values = {}

# Initialize counts for all columns to 0
for col_name in train_dataset.column_names:
    missing_values[col_name] = 0

# Iterate through each item (row) in the dataset
# Using len(train_dataset) as the dataset is not a pandas DataFrame but a datasets.Dataset object
for i in range(len(train_dataset)):
    row = train_dataset[i]
    for col_name in train_dataset.column_names:
        # Check if the value for the current column in the current row is None
        if row[col_name] is None:
            missing_values[col_name] += 1

print("Missing values in train_dataset by column:")
for col, count in missing_values.items():
    print(f"- {col}: {count}")

In [ ]:
safe_data = train_dataset.filter(lambda example: example['is_safe'] == True)
unsafe_data = train_dataset.filter(lambda example: example['is_safe'] == False)

print("First 5 entries of SAFE data:")
display(safe_data.select(range(5)).to_pandas())

print("\nFirst 5 entries of UNSAFE data:")
display(unsafe_data.select(range(5)).to_pandas())

In [ ]:
print("Analyzing subcategories in SAFE data:")
safe_category_counts = {}

# Filter for safe data first
safe_data = train_dataset.filter(lambda example: example['is_safe'] == True)

# Iterate through the 'category' column of the SAFE data to count occurrences of True for each sub-category
for i in range(len(safe_data)):
    row_categories = safe_data[i]['category']
    for sub_category, is_true in row_categories.items():
        if is_true:
            safe_category_counts[sub_category] = safe_category_counts.get(sub_category, 0) + 1

# Sort categories by count for better visualization
sorted_safe_category_counts = dict(sorted(safe_category_counts.items(), key=lambda item: item[1], reverse=True))

# Print the distribution of each sub-category in SAFE data
if sorted_safe_category_counts:
    print("\nDistribution of sub-categories in SAFE data (where value is True):")
    for cat, count in sorted_safe_category_counts.items():
        print(f"- {cat}: {count}")
else:
    print("\nNo specific sub-categories were marked as 'True' for any entry in the SAFE dataset. This indicates that content marked as 'safe' generally does not trigger any of the defined problematic sub-categories.")

In [ ]:
import pandas as pd

def extract_features(example):
    # Extract 'is_safe' as an integer
    example['is_safe_int'] = int(example['is_safe'])

    # Extract each subcategory as a separate binary feature
    for sub_category_name in train_dataset.features['category'].keys():
        example[sub_category_name] = int(example['category'].get(sub_category_name, False))
    return example

# Apply the function to the entire train_dataset
train_dataset_engineered = train_dataset.map(extract_features)

# Display the first few entries with the new features
print("First 5 entries of the engineered dataset with new features:")
display(train_dataset_engineered.select(range(5)).to_pandas())

In [ ]:
print("Analyzing safety distribution for each subcategory:")
for sub_category_name in train_dataset.features['category'].keys():
    # Filter data for the current subcategory where its value is True
    subcategory_data = train_dataset.filter(lambda example: example['category'].get(sub_category_name, False) == True)

    if len(subcategory_data) > 0:
        is_safe_counts_subcategory = subcategory_data.to_pandas()['is_safe'].value_counts()
        print(f"\n--- Subcategory: '{sub_category_name}' ---")
        print(f"Distribution of 'is_safe':\n{is_safe_counts_subcategory}")

        # Determine if it's generally safe or unsafe
        if True in is_safe_counts_subcategory and False in is_safe_counts_subcategory:
            if is_safe_counts_subcategory[False] > is_safe_counts_subcategory[True]:
                print(f"Based on the data, the '{sub_category_name}' subcategory is predominantly UNSAFE.")
            else:
                print(f"Based on the data, the '{sub_category_name}' subcategory is predominantly SAFE.")
        elif True in is_safe_counts_subcategory:
            print(f"Based on the data, all entries in the '{sub_category_name}' subcategory are SAFE.")
        elif False in is_safe_counts_subcategory:
            print(f"Based on the data, all entries in the '{sub_category_name}' subcategory are UNSAFE.")
    else:
        print(f"\n--- Subcategory: '{sub_category_name}' ---")
        print("No entries found for this subcategory.")

### Analyzing Label Distribution



value counts of the 'is_safe' column and visually representing these counts



In [ ]:
import matplotlib.pyplot as plt

# 1. Calculate the value counts for the 'is_safe' column
is_safe_counts = train_dataset.to_pandas()['is_safe'].value_counts()

# 2. Print the distribution of 'is_safe' values
print("Distribution of 'is_safe' values:")
print(is_safe_counts)
print("\n")

# 3. Create a bar chart to visualize the distribution of 'is_safe' values
plt.figure(figsize=(6, 4))
is_safe_counts.plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title("Distribution of 'is_safe' Column")
plt.xlabel("Is Safe")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
category_counts = {}

# 4. Iterate through the 'category' column to count the occurrences of True for each sub-category
# The 'category' column is a dictionary where keys are sub-category names and values are booleans.
for i in range(len(train_dataset)): # Iterate through each row
    row_categories = train_dataset[i]['category']
    for sub_category, is_true in row_categories.items():
        if is_true:
            category_counts[sub_category] = category_counts.get(sub_category, 0) + 1

# Sort categories by count for better visualization
sorted_category_counts = dict(sorted(category_counts.items(), key=lambda item: item[1], reverse=True))

# 5. Print the distribution of each sub-category
print("\nDistribution of sub-categories (where value is True):")
for cat, count in sorted_category_counts.items():
    print(f"- {cat}: {count}")

# 6. Bar chart to visualize the distribution of these sub-categories
plt.figure(figsize=(12, 7)) # Adjust figure size for better readability
plt.bar(sorted_category_counts.keys(), sorted_category_counts.values(), color='lightgreen')
plt.title("Distribution of 'Category' Sub-categories")
plt.xlabel("Sub-category")
plt.ylabel("Number of Samples (True)")
plt.xticks(rotation=45, ha='right') # Rotate labels for better fit
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()

### Analyzed Text Length Distribution
Analyzed and visualized the distribution of text lengths for relevant text fields in the `train_dataset`.


Analyzed the text length distribution, I calculated the length of the 'prompt' and 'response' columns and added them as new features to the `train_dataset`. This is a preprocessing step before calculating statistics and visualization.



In [ ]:
def calculate_length(example):
    example['prompt_length'] = len(example['prompt'])
    example['response_length'] = len(example['response'])
    return example

train_dataset = train_dataset.map(calculate_length)

print("New features added: 'prompt_length' and 'response_length'.")
print(f"First 5 entries with new lengths:\n{train_dataset.select(range(5))['prompt_length']}\n{train_dataset.select(range(5))['response_length']}")

In [ ]:
import pandas as pd

# Convert to pandas DataFrame for easy descriptive statistics calculation
df_lengths = train_dataset.to_pandas()[['prompt_length', 'response_length']]

print("Descriptive statistics for 'prompt_length':")
print(df_lengths['prompt_length'].describe())
print("\n")

print("Descriptive statistics for 'response_length':")
print(df_lengths['response_length'].describe())

In [ ]:
import matplotlib.pyplot as plt

# Histograms for 'prompt_length'
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(df_lengths['prompt_length'], bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of Prompt Lengths')
plt.xlabel('Prompt Length')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Histograms for 'response_length'
plt.subplot(1, 2, 2)
plt.hist(df_lengths['response_length'], bins=50, color='lightcoral', edgecolor='black')
plt.title('Distribution of Response Lengths')
plt.xlabel('Response Length')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### Summary
Insights into data structure, missing data, and feature distributions.


## Summary:

### Data Analysis Key Findings

*   The `train_dataset` consists of 300,567 rows and 4 columns, namely `prompt`, `response`, `category`, and `is_safe`.
*   The `prompt` and `response` columns are of string type. The `category` column is a dictionary of boolean values for various sub-categories, while `is_safe` is a boolean.
*   There are no missing values (`None`) identified across any of the columns in the `train_dataset`.
*   The dataset shows an imbalance in the `is_safe` label distribution: 166,382 samples are marked as `False` (unsafe), while 134,185 samples are marked as `True` (safe).
*   Among the sub-categories within the `category` column, `violence,aiding_and_abetting,incitement` is the most frequent with 79,544 occurrences, followed by `non_violent_unethical_behavior` (59,992). Conversely, `child_abuse` (1,664), `self_harm` (2,024), and `terrorism,organized_crime` (2,457) are the least frequent.
*   Prompt lengths range from 1 to 1,029 characters, with an average length of 66.63 characters. The distribution is right-skewed, indicating most prompts are relatively short.
*   Response lengths range from 1 to 2,314 characters, with an average length of 349.05 characters. Responses are generally much longer than prompts and also exhibit a right-skewed distribution.


In [ ]:
non_violent_unethical_behavior_data = train_dataset.filter(lambda example: example['category'].get('non_violent_unethical_behavior', False) == True)

if len(non_violent_unethical_behavior_data) > 0:
    is_safe_counts_specific_category = non_violent_unethical_behavior_data.to_pandas()['is_safe'].value_counts()
    print(f"Distribution of 'is_safe' for 'non_violent_unethical_behavior' category:")
    print(is_safe_counts_specific_category)

    # Determine if it's generally safe or unsafe
    if True in is_safe_counts_specific_category and False in is_safe_counts_specific_category:
        if is_safe_counts_specific_category[False] > is_safe_counts_specific_category[True]:
            print("\nBased on the data, the 'non_violent_unethical_behavior' category is predominantly UNSAFE.")
        else:
            print("\nBased on the data, the 'non_violent_unethical_behavior' category is predominantly SAFE.")
    elif True in is_safe_counts_specific_category:
        print("\nBased on the data, all entries in the 'non_violent_unethical_behavior' category are SAFE.")
    elif False in is_safe_counts_specific_category:
        print("\nBased on the data, all entries in the 'non_violent_unethical_behavior' category are UNSAFE.")
else:
    print("No entries found for the 'non_violent_unethical_behavior' category.")